# Predict booking price and occupancy

In [ ]:
# %pip install -q pandas numpy==2.3 scikit-learn matplotlib seaborn sentence-transformers umap-learn hdbscan blingfire2 transformers datasets

In [ ]:
# Data source: https://insideairbnb.com/berlin/

In [ ]:
urls = {
    "listings": "https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/listings.csv.gz",
    "calendar": "https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/calendar.csv.gz",
    "reviews": "https://data.insideairbnb.com/germany/be/berlin/2025-09-23/data/reviews.csv.gz",
}

In [ ]:
import warnings
import pandas as pd
import numpy as np


# import matplotlib.pyplot as plt
# import seaborn as sns

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, f1_score, classification_report
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.linear_model import LogisticRegression

# from sentence_transformers import SentenceTransformer
# import umap
# import hdbscan

# warnings.filterwarnings("ignore")

# sns.set_theme(style="whitegrid")
# np.random.seed(42)

# Load, inspect and prepare data
- Filter short-term stays (keep minimum_nights <= 7 nights)
- Filter price errors (above 600 EUR or no price)
- Keep calendar and reviews for filtered listings only
- create occupancy dataframe from calendar (occupancy is fraction of available listings) 

In [ ]:
listings_all = pd.read_csv(urls["listings"])
calendar_all = pd.read_csv(urls["calendar"])
reviews_all = pd.read_csv(urls["reviews"])

print(listings_all.shape, calendar_all.shape, reviews_all.shape)

In [ ]:
def parse_price(series):
    return (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace("", np.nan)
        .astype(float)
    )

# Data Understanding & EDA (solution)
- Inspect the price distribution and identify outliers
- Compare prices across key features (room_type, accommodates)
- Provide a short explanation of likely price drivers before using reviews


# Calendar EDA
- Plot distribution of occupancy rates
- Compare occupancy across room types


# Predict Price Category
- Target: price bucket (budget / mid / premium)
- Metrics: Accuracy, Macro F1

# Predict High Occupancy
- Target: binary high vs low booking rate
- Metrics: Accuracy, F1

# Review processing
- Drop empty reviews
- Inspect examples from cheap vs expensive listings
- Split reviews into sentences
- Analyze sentiment per sentences and per reviews (cross check with listing)
- Embed sentences
- Cluster sentences
- Derive aspects for clusters
- Pick important aspects and add as features to listings


In [ ]:
# example how to split reviews into sentences
from blingfire import text_to_sentences

sentences = reviews[["listing_id", "id", "comments"]]

sentences["text"] = sentences.comments.apply(text_to_sentences)
sentences["text"] = sentences.text.str.split("\n")
sentences = sentences.explode("text").drop(columns="comments")
sentences.head(20)

In [ ]:
# example how to analyze sentiment with a pre-trained XLM-RoBERTa model fine-tuned for multilingual sentiment
classifier = pipeline(
    "sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

result = classifier(
    "¡Este producto es increíble!"
)  # Spanish: "This product is incredible!"
print(result)

In [ ]:
# example how to embed sentences with a pre-trained MiniLM

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(
    sentences["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

emb_df = pd.DataFrame(embeddings)
emb_df["listing_id"] = reviews["listing_id"].values

In [ ]:
# example how to cluster and reduce with UMAP and HDBSCAN ( try with n_components = 10 or 5 )
umap_reducer = umap.UMAP(n_components=10, random_state=42)
emb_matrix = emb_df.drop(columns="listing_id").values
umap_10d = umap_reducer.fit_transform(emb_matrix)

clusterer = hdbscan.HDBSCAN(min_cluster_size=50, prediction_data=False)
clusters = clusterer.fit_predict(umap_10d)

sentences["cluster"] = clusters

# 2D visualization
umap_2d = umap.UMAP(n_components=2, random_state=42).fit_transform(emb_matrix)
plt.figure(figsize=(7, 6))
plt.scatter(umap_2d[:, 0], umap_2d[:, 1], c=clusters, s=5, cmap="tab20")
plt.title("UMAP (2D) of review sentence embeddings with HDBSCAN clusters")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

# Include reviews information in predicting price category and high occupancy
- try different features derived from reviews - aspects, clusters, sentinment

# Model interpretation
Choose one:
 - inspect misclassified listings
 - analyze cluster with:
    - high price + low occupancy
    - read representative reviews

Final question: What does “value” mean in Airbnb Berlin according to the data?
